# Test

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def inspect_chunk_boundaries(chunks, context_words=5):
    """
    Prints the end of each chunk and the start of the next to visualize splits.
    """
    print(f"{'--- CHUNK BOUNDARY INSPECTOR ---':^60}")
    print(f"{'End of Previous Chunk':>28} | {'Start of Next Chunk':<28}")
    print("-" * 60)
    
    for i in range(len(chunks) - 1):
        # Extract the last few words of current chunk
        prev_tail = " ".join(chunks[i].split()[-context_words:])
        # Extract the first few words of next chunk
        next_head = " ".join(chunks[i+1].split()[:context_words])
        
        print(f"{i:>2}....{prev_tail:>22} | {next_head:<22}...")

# Example Technical Text
example_text = """
The Transformer model relies on Self-Attention mechanisms. 
Self-attention allows the model to weigh the importance of different words in a sequence. 
This is mathematically achieved through Query, Key, and Value matrices. 
By calculating the dot product of Query and Key, we get an attention score. 
Positional encodings are necessary because transformers lack the inherent sequential nature of RNNs. 
The Adam optimizer adaptively tunes the learning rate, which is why it is preferred for deep learning engineering.
"""

# 1. Create Chunks (Small size for visualization purposes)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=120, 
    chunk_overlap=20, # Note: Overlap will cause words to repeat at boundaries
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_text(example_text)

# 2. Inspect boundaries
inspect_chunk_boundaries(chunks, context_words=4)

/home/kevin/git_repos/dat255-teaching-assistant/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


              --- CHUNK BOUNDARY INSPECTOR ---              
       End of Previous Chunk | Start of Next Chunk         
------------------------------------------------------------
 0....relies on Self-Attention mechanisms. | Self-attention allows the model...
 1....  words in a sequence. | This is mathematically achieved...
 2....Key, and Value matrices. | By calculating the dot...
 3....get an attention score. | Positional encodings are necessary...
 4....sequential nature of RNNs. | The Adam optimizer adaptively...


In [17]:
def inspect_clean_boundaries(chunks, context_chars=50):
    """
    Shows chunk boundaries as a continuous flow without duplicating overlap.
    The boundary '|' marks where the NEXT chunk begins,
    even though part of it overlaps with the previous.
    """
    print(f"{'--- CLEAN CHUNK FLOW VIEW ---':^80}")
    print("-" * 80)

    for i in range(len(chunks) - 1):
        prev_chunk = chunks[i]
        next_chunk = chunks[i + 1]

        # Find true overlap
        overlap = ""
        max_len = min(len(prev_chunk), len(next_chunk))
        for size in range(max_len, 0, -1):
            if prev_chunk[-size:] == next_chunk[:size]:
                overlap = prev_chunk[-size:]
                break

        # Remove overlap from next chunk (so it's not duplicated)
        next_unique = next_chunk[len(overlap):]

        # Take context window
        left = (prev_chunk[-context_chars:])
        right = (next_unique[:context_chars])

        # print(f"{i:>2}.. {left}|{right}")
        print(f"{i:>2}.. {left}[{overlap}]|{right}")

        
# Example Technical Text
example_text = """
The Transformer model relies on Self-Attention mechanisms. 
Self-attention allows the model to weigh the importance of different words in a sequence. 
This is mathematically achieved through Query, Key, and Value matrices. 
By calculating the dot product of Query and Key, we get an attention score. 
Positional encodings are necessary because transformers lack the inherent sequential nature of RNNs. 
The Adam optimizer adaptively tunes the learning rate, which is why it is preferred for deep learning engineering.
"""

# 1. Create Chunks (Small size for visualization purposes)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=120, 
    chunk_overlap=20, # Note: Overlap will cause words to repeat at boundaries
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_text(example_text)

# 2. Inspect boundaries
inspect_clean_boundaries(chunks)

                         --- CLEAN CHUNK FLOW VIEW ---                          
--------------------------------------------------------------------------------
 0.. sformer model relies on Self-Attention mechanisms.[]|Self-attention allows the model to weigh the impor
 1.. h the importance of different words in a sequence.[]|This is mathematically achieved through Query, Key
 2.. y achieved through Query, Key, and Value matrices.[]|By calculating the dot product of Query and Key, w
 3.. oduct of Query and Key, we get an attention score.[]|Positional encodings are necessary because transfo
 4.. rmers lack the inherent sequential nature of RNNs.[]|The Adam optimizer adaptively tunes the learning r


In [7]:
def find_overlap(a, b, max_overlap=100):
    """
    Finds the longest suffix of 'a' that matches a prefix of 'b'.
    """
    for size in range(max_overlap, 0, -1):
        if a[-size:] == b[:size]:
            return a[-size:]
    return ""

def inspect_true_overlap(chunks, context_words=5):
    print(f"{'--- TRUE OVERLAP DETECTOR ---':^80}")
    print(f"{'Prev Tail':>25} | {'Detected Overlap':^20} | {'Next Head':<25}")
    print("-" * 80)
    
    for i in range(len(chunks) - 1):
        prev_chunk = chunks[i]
        next_chunk = chunks[i + 1]

        prev_tail = " ".join(prev_chunk.split()[-context_words:])
        next_head = " ".join(next_chunk.split()[:context_words])

        overlap = find_overlap(prev_chunk, next_chunk)

        print(f"{i:>2}.. {prev_tail:>22} | {overlap:^20} | {next_head:<22}")
# Example Technical Text
example_text = """
The Transformer model relies on Self-Attention mechanisms. 
Self-attention allows the model to weigh the importance of different words in a sequence. 
This is mathematically achieved through Query, Key, and Value matrices. 
By calculating the dot product of Query and Key, we get an attention score. 
Positional encodings are necessary because transformers lack the inherent sequential nature of RNNs. 
The Adam optimizer adaptively tunes the learning rate, which is why it is preferred for deep learning engineering.
"""

# 1. Create Chunks (Small size for visualization purposes)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=120, 
    chunk_overlap=20, # Note: Overlap will cause words to repeat at boundaries
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_text(example_text)

# 2. Inspect boundaries
inspect_true_overlap(chunks, context_words=4)

                         --- TRUE OVERLAP DETECTOR ---                          
                Prev Tail |   Detected Overlap   | Next Head                
--------------------------------------------------------------------------------
 0.. relies on Self-Attention mechanisms. |                      | Self-attention allows the model
 1..   words in a sequence. |                      | This is mathematically achieved
 2.. Key, and Value matrices. |                      | By calculating the dot
 3.. get an attention score. |                      | Positional encodings are necessary
 4.. sequential nature of RNNs. |                      | The Adam optimizer adaptively


In [18]:
# Define color and reset codes
RED = '\033[91m'
GREEN = '\033[92m'
BLUE = '\033[94m'
ENDC = '\033[0m' # Reset to default color

# Print colored text
print(RED + "This text is red!" + ENDC)
print(GREEN + "This text is green!" + ENDC)
print(BLUE + "This text is blue!" + ENDC)


This text is red!
This text is green!
This text is blue!


In [ ]:
def inspect_chunk_overlap_words(chunks, context_words=5):
    """
    Displays chunk boundaries as:
    [before context] | [overlap] | [after context]

    - No duplicated text outside overlap
    - Word-based context window
    - Color-coded output for Jupyter
    """

    # ANSI colors
    RED = '\033[91m'     # before
    GREEN = '\033[92m'   # overlap
    BLUE = '\033[94m'    # after
    ENDC = '\033[0m'

    print(f"{'--- CHUNK OVERLAP (WORD-LEVEL) ---':^100}")
    print("-" * 100)

    def find_word_overlap(a_words, b_words):
        """Find longest suffix of a that matches prefix of b (word-level)."""
        max_len = min(len(a_words), len(b_words))
        for size in range(max_len, 0, -1):
            if a_words[-size:] == b_words[:size]:
                return size
        return 0

    for i in range(len(chunks) - 1):
        a_words = chunks[i].split()
        b_words = chunks[i + 1].split()

        overlap_size = find_word_overlap(a_words, b_words)

        overlap_words = a_words[-overlap_size:] if overlap_size > 0 else []
        before_words = a_words[-(overlap_size + context_words):-overlap_size] if overlap_size > 0 else a_words[-context_words:]
        after_words = b_words[overlap_size:overlap_size + context_words]

        before_text = " ".join(before_words)
        overlap_text = " ".join(overlap_words)
        after_text = " ".join(after_words)

        # Colored output
        print(
            f"{i:>2}.. "
            f"{RED}{before_text}{ENDC} | "
            f"{GREEN}{overlap_text}{ENDC} | "
            f"{BLUE}{after_text}{ENDC}"
        )
        print()  # extra spacing for readability
        
# 1. Create Chunks (Small size for visualization purposes)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=120, 
    chunk_overlap=20, # Note: Overlap will cause words to repeat at boundaries
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_text(example_text)
inspect_chunk_overlap_words(chunks, context_words=4)


# Not using token based splitter:
#If you actually want true word overlap:

#from langchain.text_splitter import TokenTextSplitter

#splitter = TokenTextSplitter(
#    chunk_size=40,
#    chunk_overlap=10
#)


# Now overlap is token-based → your function will work as expected.

NameError: name 'RecursiveCharacterTextSplitter' is not defined

In [2]:
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter

# def normalize(words):
#     return [re.sub(r'[^\w]', '', w).lower() for w in words]

# def find_word_overlap(a_words, b_words):
#     a_norm = normalize(a_words)
#     b_norm = normalize(b_words)

#     max_len = min(len(a_norm), len(b_norm))
#     for size in range(max_len, 0, -1):
#         if a_norm[-size:] == b_norm[:size]:
#             return size
#     return 0

def inspect_chunk_overlap_words2(chunks, context_words=5):
    """
    Displays chunk boundaries as:
    [before context] | [overlap] | [after context]

    - No duplicated text outside overlap
    - Word-based context window
    - Color-coded output for Jupyter
    """

    # ANSI colors
    RED = '\033[91m'     # before
    GREEN = '\033[92m'   # overlap
    BLUE = '\033[94m'    # after
    ENDC = '\033[0m'

    print(f"{'--- CHUNK OVERLAP (WORD-LEVEL) ---':^100}")
    print("-" * 100)

    def normalize(words):
        return [re.sub(r'[^\w]', '', w).lower() for w in words]

    def find_word_overlap(a_words, b_words):
        a_norm = normalize(a_words)
        b_norm = normalize(b_words)

        max_len = min(len(a_norm), len(b_norm))
        for size in range(max_len, 0, -1):
            if a_norm[-size:] == b_norm[:size]:
                return size
        return 0

    for i in range(len(chunks) - 1):
        a_words = chunks[i].split()
        b_words = chunks[i + 1].split()

        overlap_size = find_word_overlap(a_words, b_words)

        overlap_words = a_words[-overlap_size:] if overlap_size > 0 else []
        before_words = a_words[-(overlap_size + context_words):-overlap_size] if overlap_size > 0 else a_words[-context_words:]
        after_words = b_words[overlap_size:overlap_size + context_words]

        before_text = " ".join(before_words)
        overlap_text = " ".join(overlap_words)
        after_text = " ".join(after_words)

        # Colored output
        print(
            f"{i:>2}.. "
            f"{RED}{before_text}{ENDC} | "
            f"{GREEN}{overlap_text}{ENDC} | "
            f"{BLUE}{after_text}{ENDC}"
        )
        print()  # extra spacing for readability

# Example Technical Text
example_text = """
The Transformer model relies on Self-Attention mechanisms. 
Self-attention allows the model to weigh the importance of different words in a sequence. 
This is mathematically achieved through Query, Key, and Value matrices. 
By calculating the dot product of Query and Key, we get an attention score. 
Positional encodings are necessary because transformers lack the inherent sequential nature of RNNs. 
The Adam optimizer adaptively tunes the learning rate, which is why it is preferred for deep learning engineering.
"""

# 1. Create Chunks (Small size for visualization purposes)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, 
    chunk_overlap=20, # Note: Overlap will cause words to repeat at boundaries
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_text(example_text)
print("chunk size: 200, chunk overlap: 20")
inspect_chunk_overlap_words2(chunks, context_words=4)


# 1. Create Chunks (Small size for visualization purposes)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, 
    chunk_overlap=100, # Note: Overlap will cause words to repeat at boundaries
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_text(example_text)
print("\n\nchunk size: 200, chunk overlap: 100")
inspect_chunk_overlap_words2(chunks, context_words=4)

chunk size: 200, chunk overlap: 20
                                 --- CHUNK OVERLAP (WORD-LEVEL) ---                                 
----------------------------------------------------------------------------------------------------
 0.. words in a sequence. |  | This is mathematically achieved

 1.. get an attention score. |  | Positional encodings are necessary

 2.. sequential nature of RNNs. |  | The Adam optimizer adaptively



chunk size: 200, chunk overlap: 100
                                 --- CHUNK OVERLAP (WORD-LEVEL) ---                                 
----------------------------------------------------------------------------------------------------
 0.. relies on Self-Attention mechanisms. | Self-attention allows the model to weigh the importance of different words in a sequence. | This is mathematically achieved

 1.. words in a sequence. | This is mathematically achieved through Query, Key, and Value matrices. | By calculating the dot

 2.. Key, and Value matric

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import hnswlib

splitter = RecursiveCharacterTextSplitter(
    chunk_size=120, 
    chunk_overlap=20, # Note: Overlap will cause words to repeat at boundaries
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_text(example_text)


# 1. Initialize SOTA local embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# 1. Embed chunks
# chunk_texts = [c["text"] for c in chunks]
chunk_texts = chunks
embeddings = embed_model.encode(chunk_texts, normalize_embeddings=True)

# 2. Initialize hnswlib index
dim = embeddings.shape[1]
num_elements = len(embeddings)
p = hnswlib.Index(space='cosine', dim=dim) 
p.init_index(max_elements=num_elements, ef_construction=200, M=16)
p.add_items(embeddings, np.arange(num_elements))

# 3. Standalone Retrieval Test
query = "How do transformers handle word order?"
query_embedding = embed_model.encode([query], normalize_embeddings=True)

labels, distances = p.knn_query(query_embedding, k=2)
print(f"Query: {query}")
for idx in labels[0]:
    print(f"\n{chunks[int(idx)]}")
    # print(f"\n{chunks[int(idx)]['source']}")
    # print(f"\n{chunks[int(idx)]['source']}")
    
    # print(f"Retrieved Chunk: {chunks[int(idx)]['text']}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7920.89it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query: How do transformers handle word order?

Positional encodings are necessary because transformers lack the inherent sequential nature of RNNs.

The Transformer model relies on Self-Attention mechanisms.


In [ ]:
# Test on chapter 1 of deep learning book:
data_path = "C:\Users\k97ev\Documents\git_repos\dat255-teaching-assistant\chapters_chapter01_what-is-deep-learning.md"
with open(data_path, "r", encoding="utf-8") as f:
    chapter_text = f.read()